In [ ]:
import re
import os
import glob
import pandas as pd
import numpy as np
import nonlinearestimator as nle
from tqdm import tqdm

In [ ]:
baseFolder = "/mnt/DATA/NonLinearMI/"
candidates = []
with open(os.path.join(baseFolder,"possible.txt")) as fp:
    for line in fp:
        folderName = line.strip()
        bins = re.findall(r"bin([\d]*)", folderName)[0]
        if os.path.isfile(os.path.join(baseFolder,folderName,f"patient00_{bins}.npy")):# and os.path.isfile(os.path.join(baseFolder,folderName,f"patient00_{bins}_cor.npy")):
            candidates.append(folderName)
        else:
            print("Nothing to do for", folderName)


In [ ]:
len(candidates)

In [ ]:
def checkSame (firstSet, secondSet):
    firstNums = sorted([int(x.split("nt")[1]) for x in firstSet])
    if firstSet == secondSet:
        return True, firstNums[-1]
    else:
        secondNums = sorted([int(x.split("nt")[1]) for x in secondSet])
        for i in range(max(len(firstNums), len(secondNums))):
            try:
                if firstNums[i] != secondNums[i]:
                    return False, firstNums[i-1]
            except IndexError:
                return False, firstNums[i-1]
        print("Weird!", firstSet, secondSet, sep="\n")
        return True, firstNums[-1]


deleting = []
for folderName in candidates:
    thisFolderPlan = [folderName]
    bins = re.findall(r"bin([\d]*)", folderName)[0]
    MIfiles = set(x.split("/")[-1].split("_")[0] for x in  glob.glob(os.path.join(baseFolder,folderName,f"patient*_{bins}.npy")))
    corfiles = set(x.split("/")[-1].split("_")[0] for x in  glob.glob(os.path.join(baseFolder,folderName,f"patient*_{bins}_cor.npy")))
    PDFfiles = set(x.split("/")[-1].split("_")[0] for x in  glob.glob(os.path.join(baseFolder,folderName,f"patient*_{bins}.pdf")))
    thisFolderPlan.extend(checkSame(MIfiles, corfiles))
    thisFolderPlan.extend(checkSame(MIfiles, PDFfiles))
    thisFolderPlan.append(max(map(len,[MIfiles,corfiles,PDFfiles])))
    deleting.append(thisFolderPlan)
del_df = pd.DataFrame(deleting, columns=["folder", "allCor", "numCor", "allPdf", "numPdf", "totSub"])
del_df

In [ ]:
(del_df.totSub-del_df.numCor-1).values, (del_df.totSub-del_df.numPdf-1).values

In [ ]:
datasetNames = {"EEG_bands_band":"EEG_bands", "boldEso":"boldEso190", "EEG_bands_band1_sub":"EEG_bands_sub", "electrode_BLP_band":"electrode_BLP", "source_BLP_band":"source_BLP", "eso245_cra_strin_":"eso245_cra_strin", "ESO190_strin_AAL":"ESO190_strin_AAL90","MS_fMRI":"MS_fMRI", "movz_aal_mod_":"movz_aal", "ID09_Sz6ort_band":None, "ID09_Sz6raw_band":None, "iEEG_ID12_band":None, "iEEG_part_band":"iEEG","iEEG_part":"iEEG_all","~MS_fMRI":None}
final = []
for folder in del_df.folder:
    forFinal = [folder]
    fip, bins = folder.split("_bin")
    try:
        reg = re.findall(r"([\d]+)", fip[-4:])[0]
        name = fip[:-int(np.log10(int(reg)))-1]
    except IndexError:
        reg=""
        name = fip
    if name.startswith("eso245_aal"):
        forFinal.append("eso245_aal")
        forFinal.append(name.split("_")[-2])
    else:
        forFinal.append(datasetNames[name])
        forFinal.append(reg)
    forFinal.append(bins)
    final.append(forFinal)
final_df = pd.DataFrame(final, columns=["folder","dataset","regions","bins"])
print(final_df)

In [ ]:
# corDel=0
# pdfDel=0
# for folder in tqdm(final_df.iterrows(), total=len(final_df)):
#     for file in glob.glob(os.path.join(baseFolder,folder[1].folder,f"patient*_{folder[1].bins}.pdf")):
#         pdfDel += 1
#         os.unlink(file)
#     for file in glob.glob(os.path.join(baseFolder,folder[1].folder,f"patient*_{folder[1].bins}_cor.npy")):
#         corDel += 1
#         os.unlink(file)
# print(f"Deleted {pdfDel} pdfs and {corDel} cors")

In [ ]:
final_df.drop(91, inplace=True)

In [ ]:
for i, folder in tqdm(final_df.iterrows(), total=len(final_df)):
    print(folder.dataset, folder.regions, folder.bins)
    if folder.dataset is None:
        continue
    if os.path.isfile(os.path.join(baseFolder,folder.folder,f"patient00_{folder.bins}_cor.npy")):
        print(folder.folder, "already there.")
        continue
    estimator = nle.NonLinearEstimator(dataset=folder.dataset, nbins=folder.bins, regions=folder.regions, savenpy=True)
    estimator.run()
    del estimator